In [1]:
import torch
torch.cuda.empty_cache()


In [2]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import numpy as np
from tqdm.auto import tqdm
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# Add parent directories to path
script_dir = Path.cwd()
sys.path.insert(0, str(script_dir.parent.parent))

from config import (
    PROJECT_ROOT,
    SAMPLING_RATE,
    XLSR_MAX_TIME_STEPS
)
from model import SSLModel, AD_XLSR_Model
from model_DANN import GradientReversalLayer, compute_alpha
from data_split import create_train_val_split
from visualization import plot_training_curves, plot_dataset_comparison


## Configuration


In [3]:
SOURCE_DATASET = "Pitt"
TARGET_DATASET = "Lu"

# Data paths - following the same pattern as frozen-DANN-ADReSS
data_dir = PROJECT_ROOT / "data" / "processed"
raw_data_dir = PROJECT_ROOT / "data" / "raw"

SOURCE_RAW_AUDIO_DIR = raw_data_dir / SOURCE_DATASET
TARGET_RAW_AUDIO_DIR = raw_data_dir / TARGET_DATASET

SOURCE_TRAIN_CSV = data_dir / f"{SOURCE_DATASET}-xlsr-train.csv"
SOURCE_VAL_CSV = data_dir / f"{SOURCE_DATASET}-xlsr-val.csv"
TARGET_TRAIN_CSV = data_dir / f"{TARGET_DATASET}-xlsr-train.csv"
TARGET_VAL_CSV = data_dir / f"{TARGET_DATASET}-xlsr-val.csv"

MODEL_OUTPUT_DIR = PROJECT_ROOT / "models" / f"FINETUNED_DANN_{SOURCE_DATASET}_to_{TARGET_DATASET}_xlsr"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
# Training hyperparameters
AUDIO_LENGTH_SEC = 60
FINETUNE_BATCH_SIZE = 4
ACCUMULATION_STEPS = 4
XLSR_LR = 1e-5              # Learning rate for XLSR fine-tuned layers
CLASSIFIER_LR = 1e-3        # Learning rate for classifier
DOMAIN_CLASSIFIER_LR = 1e-3 # Learning rate for domain classifier
FINETUNE_MAX_EPOCHS = 50
FINETUNE_PATIENCE = 15
NUM_WORKERS = 0

# Model architecture
FINETUNE_LAYERS_NUM = 3
XLSR_DROPOUT = 0.4
WEIGHT_DECAY = 5e-2

# Class weights
CLASS_WEIGHT_CONTROL = 1.5
CLASS_WEIGHT_DEMENTIA = 1.0

# DANN parameters
LAMBDA_CLASS = 1.0
LAMBDA_DOMAIN = 1.0
DOMAIN_WARMUP_EPOCHS = 10   # First N epochs: no domain adversarial loss
DOMAIN_ANNEAL_RATIO = 0.25  # Next 25% epochs: linearly increase lambda

# Model selection weights
SOURCE_WEIGHT = 0.5
TARGET_WEIGHT = 0.5
MIN_SOURCE_ACC = 0.60
MIN_TARGET_ACC = 0.60


In [5]:
# Device setup
if torch.cuda.is_available():
    device = torch.device('cuda')
    accelerator = 'gpu'
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    accelerator = 'mps'
else:
    device = torch.device('cpu')
    accelerator = 'cpu'
    
print(f"Using device: {device}")


Using device: mps


## Data Preparation


In [6]:
# Create source dataset splits
SOURCE_TRAIN_CSV, SOURCE_VAL_CSV = create_train_val_split(
    raw_audio_dir=SOURCE_RAW_AUDIO_DIR,
    train_csv_path=SOURCE_TRAIN_CSV,
    val_csv_path=SOURCE_VAL_CSV,
    feature_dir_name=f"{SOURCE_DATASET}_xlsr_features",
    dataset_name=SOURCE_DATASET,
    xlsr=True
)

# Create target dataset splits
TARGET_TRAIN_CSV, TARGET_VAL_CSV = create_train_val_split(
    raw_audio_dir=TARGET_RAW_AUDIO_DIR,
    train_csv_path=TARGET_TRAIN_CSV,
    val_csv_path=TARGET_VAL_CSV,
    feature_dir_name=f"{TARGET_DATASET}_xlsr_features",
    dataset_name=TARGET_DATASET,
    xlsr=True
)


============= Pitt Train(80.0%) and Val(20%) Split Complete! =============
Training set: 440 samples (Control: 193, Dementia: 247)
Validation set: 111 samples (Control: 49, Dementia: 62)
============= Lu Train(80.0%) and Val(20%) Split Complete! =============
Training set: 58 samples (Control: 28, Dementia: 30)
Validation set: 16 samples (Control: 8, Dementia: 8)


## Dataset and DataLoader


In [7]:
class AudioDataset(Dataset):
    """Dataset for loading audio files directly for XLSR fine-tuning with DANN"""
    def __init__(self, csv_path, audio_dir, max_length_sec=60, sampling_rate=16000, return_labels=True):
        self.csv_path = csv_path
        self.audio_dir = Path(audio_dir)
        self.max_length_sec = max_length_sec
        self.sampling_rate = sampling_rate
        self.max_samples = max_length_sec * sampling_rate
        self.return_labels = return_labels
        
        # Load data from CSV
        self.data = []
        df = pd.read_csv(csv_path)
        
        for _, row in df.iterrows():
            session_id = row['session_id']
            label = int(row['ad']) if return_labels else -1
            
            label_dir = "Dementia" if label == 1 else "Control"
            
            found = False
            for ext in ['.wav', '.mp3', '.flac']:
                audio_path = self.audio_dir / label_dir / f"{session_id}{ext}"
                if audio_path.exists():
                    self.data.append({
                        'session_id': session_id,
                        'audio_path': audio_path,
                        'label': label
                    })
                    found = True
                    break
            
            if not found:
                print(f"Warning: Audio file not found for session {session_id}")
        
        print(f"Loaded {len(self.data)} audio files from {csv_path.name}")
        if return_labels:
            control_count = sum(1 for d in self.data if d['label'] == 0)
            dementia_count = sum(1 for d in self.data if d['label'] == 1)
            print(f"  Control: {control_count}, Dementia: {dementia_count}")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        audio_path = item['audio_path']
        label = item['label']
        
        # Load audio
        waveform, sr = torchaudio.load(audio_path)
        
        # Resample if necessary
        if sr != self.sampling_rate:
            resampler = torchaudio.transforms.Resample(sr, self.sampling_rate)
            waveform = resampler(waveform)
        
        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        waveform = waveform.squeeze(0)
        
        # Pad or truncate
        if waveform.shape[0] > self.max_samples:
            waveform = waveform[:self.max_samples]
        elif waveform.shape[0] < self.max_samples:
            padding = self.max_samples - waveform.shape[0]
            waveform = F.pad(waveform, (0, padding))
        
        return waveform, label


def collate_audio_batch(batch):
    waveforms = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    
    waveforms = torch.stack(waveforms)
    labels = torch.tensor(labels, dtype=torch.long)
    
    return waveforms, labels


In [8]:
# Create datasets
source_train_dataset = AudioDataset(
    csv_path=SOURCE_TRAIN_CSV, audio_dir=SOURCE_RAW_AUDIO_DIR,
    max_length_sec=AUDIO_LENGTH_SEC, sampling_rate=SAMPLING_RATE, return_labels=True
)

source_val_dataset = AudioDataset(
    csv_path=SOURCE_VAL_CSV, audio_dir=SOURCE_RAW_AUDIO_DIR,
    max_length_sec=AUDIO_LENGTH_SEC, sampling_rate=SAMPLING_RATE, return_labels=True
)

target_train_dataset = AudioDataset(
    csv_path=TARGET_TRAIN_CSV, audio_dir=TARGET_RAW_AUDIO_DIR,
    max_length_sec=AUDIO_LENGTH_SEC, sampling_rate=SAMPLING_RATE, return_labels=True
)

target_val_dataset = AudioDataset(
    csv_path=TARGET_VAL_CSV, audio_dir=TARGET_RAW_AUDIO_DIR,
    max_length_sec=AUDIO_LENGTH_SEC, sampling_rate=SAMPLING_RATE, return_labels=True
)

# Create data loaders
source_train_loader = DataLoader(
    source_train_dataset, batch_size=FINETUNE_BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_audio_batch,
    pin_memory=torch.cuda.is_available()
)

source_val_loader = DataLoader(
    source_val_dataset, batch_size=FINETUNE_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_audio_batch,
    pin_memory=torch.cuda.is_available()
)

target_train_loader = DataLoader(
    target_train_dataset, batch_size=FINETUNE_BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_audio_batch,
    pin_memory=torch.cuda.is_available()
)

target_val_loader = DataLoader(
    target_val_dataset, batch_size=FINETUNE_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_audio_batch,
    pin_memory=torch.cuda.is_available()
)

print(f"\nSource train batches: {len(source_train_loader)}")
print(f"Source val batches: {len(source_val_loader)}")
print(f"Target train batches: {len(target_train_loader)}")
print(f"Target val batches: {len(target_val_loader)}")


Loaded 440 audio files from Pitt-xlsr-train.csv
  Control: 193, Dementia: 247
Loaded 111 audio files from Pitt-xlsr-val.csv
  Control: 49, Dementia: 62
Loaded 58 audio files from Lu-xlsr-train.csv
  Control: 28, Dementia: 30
Loaded 16 audio files from Lu-xlsr-val.csv
  Control: 8, Dementia: 8

Source train batches: 110
Source val batches: 28
Target train batches: 15
Target val batches: 4


## Model Architecture: Fine-tuned XLSR + DANN


In [9]:
class XLSR_Finetune_DANN_Model(nn.Module):
    """Fine-tuned XLSR with Domain Adversarial Neural Network"""
    def __init__(self, device, dropout=0.2, num_finetune_layers=3):
        super().__init__()
        
        # XLSR feature extractor (freeze all first)
        self.ssl_model = SSLModel(device=device, freeze_xlsr=True)
        
        # Unfreeze only the last N transformer layers
        encoder_layers = self.ssl_model.model.encoder.layers
        total_layers = len(encoder_layers)
        for i, layer in enumerate(encoder_layers):
            if i >= total_layers - num_finetune_layers:
                for param in layer.parameters():
                    param.requires_grad = True
        
        self.ssl_model.model.train()
        print(f"XLSR: Fine-tuning layers {total_layers - num_finetune_layers} to {total_layers - 1}")
        
        # AD classifier (label predictor)
        self.ad_classifier = AD_XLSR_Model(dropout=dropout)
        
        # Gradient Reversal Layer
        self.grl = GradientReversalLayer()
        
        # Domain classifier
        self.domain_classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)  # Binary: source (0) vs target (1)
        )
        
        self.device = device
    
    def forward(self, waveform, alpha=1.0, return_domain=False):
        """Forward pass with optional domain classification"""
        # Extract XLSR features
        embedding, _ = self.ssl_model.extract_feat(waveform)
        batch_size, seq_len, _ = embedding.shape
        
        # Pad or truncate to XLSR_MAX_TIME_STEPS
        if seq_len > XLSR_MAX_TIME_STEPS:
            embedding = embedding[:, :XLSR_MAX_TIME_STEPS, :]
            mask = torch.ones(batch_size, XLSR_MAX_TIME_STEPS, device=self.device)
        elif seq_len < XLSR_MAX_TIME_STEPS:
            padding = torch.zeros(batch_size, XLSR_MAX_TIME_STEPS - seq_len, 1024, device=self.device)
            embedding = torch.cat([embedding, padding], dim=1)
            mask = torch.ones(batch_size, XLSR_MAX_TIME_STEPS, device=self.device)
            mask[:, seq_len:] = 0
        else:
            mask = torch.ones(batch_size, seq_len, device=self.device)
        
        # AD classification
        class_logits = self.ad_classifier(embedding, mask)
        
        if return_domain:
            # Domain classification (with gradient reversal)
            pooled_features = (embedding * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True)
            reversed_features = self.grl(pooled_features, alpha)
            domain_logits = self.domain_classifier(reversed_features)
            return class_logits, domain_logits
        
        return class_logits


## Training Functions


In [10]:
def compute_domain_lambda(epoch, total_epochs, base_lambda=LAMBDA_DOMAIN,
                         warmup_epochs=DOMAIN_WARMUP_EPOCHS, anneal_ratio=DOMAIN_ANNEAL_RATIO):
    """Schedule domain loss weight based on epoch"""
    warmup_epochs = min(warmup_epochs, total_epochs)
    anneal_epochs = int(total_epochs * anneal_ratio)
    
    if epoch <= warmup_epochs:
        return 0.0
    
    if anneal_epochs <= 0:
        return base_lambda
    
    progress = min(1.0, max(0, epoch - warmup_epochs) / max(1, anneal_epochs))
    return base_lambda * progress


def train_one_epoch_finetune_dann(model, source_loader, target_loader, optimizer, device, 
                                  epoch, total_epochs, accumulation_steps=1,
                                  class_weights=None, lambda_class=1.0, lambda_domain=1.0):
    """Train one epoch with fine-tuned XLSR and DANN"""
    model.train()
    
    total_class_loss = 0
    total_domain_loss = 0
    correct = 0
    total = 0
    domain_correct = 0
    domain_total = 0
    
    target_iter = iter(target_loader)
    len_source = len(source_loader)
    total_steps = total_epochs * len_source
    lambda_active = lambda_domain > 0
    
    pbar = tqdm(enumerate(source_loader), total=len_source, desc=f"Epoch {epoch}", leave=False)
    
    optimizer.zero_grad()
    
    for i, source_batch in pbar:
        current_step = (epoch - 1) * len_source + i
        alpha = compute_alpha(current_step, total_steps)
        
        source_waveforms, source_labels = source_batch
        source_waveforms = source_waveforms.to(device)
        source_labels = source_labels.to(device)
        
        if lambda_active:
            class_output, domain_output = model(
                source_waveforms, alpha=alpha, return_domain=True
            )
        else:
            class_output = model(
                source_waveforms, alpha=alpha, return_domain=False
            )
            domain_output = None
        
        source_domain_labels = torch.zeros(source_labels.size(0), dtype=torch.long).to(device)
        
        loss_class = F.cross_entropy(class_output, source_labels, weight=class_weights)
        
        if lambda_active:
            loss_domain_source = F.cross_entropy(domain_output, source_domain_labels)
            
            try:
                target_batch = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                target_batch = next(target_iter)
            
            target_waveforms, _ = target_batch
            target_waveforms = target_waveforms.to(device)
            
            _, domain_output_target = model(
                target_waveforms, alpha=alpha, return_domain=True
            )
            
            target_domain_labels = torch.ones(target_waveforms.size(0), dtype=torch.long).to(device)
            loss_domain_target = F.cross_entropy(domain_output_target, target_domain_labels)
        else:
            loss_domain_source = torch.zeros(1, device=device)
            loss_domain_target = torch.zeros(1, device=device)
            target_domain_labels = None
            domain_output_target = None
        
        total_batch_loss = (
            lambda_class * loss_class +
            lambda_domain * (loss_domain_source + loss_domain_target)
        )
        
        total_batch_loss = total_batch_loss / accumulation_steps
        total_batch_loss.backward()
        
        if (i + 1) % accumulation_steps == 0 or (i + 1) == len_source:
            optimizer.step()
            optimizer.zero_grad()
        
        total_class_loss += loss_class.item()
        total_domain_loss += (loss_domain_source.item() + loss_domain_target.item())
        
        predictions = torch.argmax(class_output, dim=1)
        correct += (predictions == source_labels).sum().item()
        total += source_labels.size(0)
        
        if lambda_active:
            domain_pred_source = torch.argmax(domain_output, dim=1)
            domain_pred_target = torch.argmax(domain_output_target, dim=1)
            domain_correct += (domain_pred_source == source_domain_labels).sum().item()
            domain_correct += (domain_pred_target == target_domain_labels).sum().item()
            domain_total += source_domain_labels.size(0) + target_domain_labels.size(0)
        
        pbar.set_postfix({
            'α': f'{alpha:.2f}',
            'λ_d': f'{lambda_domain:.3f}',
            'cls': f'{loss_class.item():.3f}',
            'dom': f'{(loss_domain_source.item() + loss_domain_target.item()):.3f}',
            'acc': f'{correct/total:.3f}'
        })
    
    avg_class_loss = total_class_loss / len_source
    avg_domain_loss = total_domain_loss / len_source
    train_acc = correct / total
    domain_acc = (domain_correct / domain_total) if domain_total > 0 else 0.0
    
    return avg_class_loss, avg_domain_loss, train_acc, domain_acc


def validate(model, val_loader, device, domain_name=""):
    """Validate model and compute detailed metrics"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    control_correct = 0
    control_total = 0
    dementia_correct = 0
    dementia_total = 0
    true_positives = 0
    false_positives = 0
    false_negatives = 0

    desc = f"Validating {domain_name}" if domain_name else "Validation"
    pbar = tqdm(val_loader, desc=desc, leave=False)

    with torch.no_grad():
        for waveforms, labels in pbar:
            waveforms = waveforms.to(device)
            labels = labels.to(device)
            
            logits = model(waveforms, return_domain=False)
            loss = F.cross_entropy(logits, labels)
            predictions = torch.argmax(logits, dim=1)

            total_loss += loss.item()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            for pred, label in zip(predictions, labels):
                if label == 0:
                    control_total += 1
                    if pred == label:
                        control_correct += 1
                else:
                    dementia_total += 1
                    if pred == label:
                        dementia_correct += 1

                if pred == 1 and label == 1:
                    true_positives += 1
                elif pred == 1 and label == 0:
                    false_positives += 1
                elif pred == 0 and label == 1:
                    false_negatives += 1
            
            current_loss = total_loss / (pbar.n + 1)
            current_acc = correct / total
            pbar.set_postfix({'loss': f'{current_loss:.4f}', 'acc': f'{current_acc:.4f}'})

    avg_loss = total_loss / len(val_loader)
    accuracy = correct / total
    control_acc = control_correct / control_total if control_total > 0 else 0
    dementia_acc = dementia_correct / dementia_total if dementia_total > 0 else 0

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return avg_loss, accuracy, control_acc, dementia_acc, f1_score


## Main Training Loop


In [11]:
def train_finetune_dann(seed, source_train_loader, target_train_loader, 
                        source_val_loader, target_val_loader, 
                        output_dir, device,
                        source_weight=0.5, target_weight=0.5,
                        min_source_acc=0.0, min_target_acc=0.0):
    """Complete fine-tuned DANN training pipeline"""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    np.random.seed(seed)

    seed_dir = Path(output_dir) / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    model = XLSR_Finetune_DANN_Model(
        device=device, dropout=XLSR_DROPOUT, num_finetune_layers=FINETUNE_LAYERS_NUM
    ).to(device)
    
    xlsr_trainable_params = [p for p in model.ssl_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {'params': xlsr_trainable_params, 'lr': XLSR_LR},
        {'params': model.ad_classifier.parameters(), 'lr': CLASSIFIER_LR},
        {'params': model.domain_classifier.parameters(), 'lr': DOMAIN_CLASSIFIER_LR}
    ], weight_decay=WEIGHT_DECAY)

    class_weights = torch.tensor([CLASS_WEIGHT_CONTROL, CLASS_WEIGHT_DEMENTIA], device=device)

    history = {
        'epochs': [], 'train_class_losses': [], 'train_domain_losses': [],
        'train_accs': [], 'domain_accs': [], 'source_val_accs': [],
        'target_val_accs': [], 'avg_val_accs': [], 'source_val_losses': [],
        'target_val_losses': [], 'lambda_domains': []
    }

    best_avg_acc = 0
    best_metrics = {}
    patience_counter = 0
    best_epoch = 0

    print(f"\n{'='*70}")
    print(f"Training with seed {seed}")
    print(f"Model selection: Avg acc = {source_weight:.2f}×source + {target_weight:.2f}×target")
    print(f"Minimum thresholds: Source≥{min_source_acc:.2f}, Target≥{min_target_acc:.2f}")
    print(f"{'='*70}")

    for epoch in range(1, FINETUNE_MAX_EPOCHS + 1):
        current_lambda_domain = compute_domain_lambda(epoch, FINETUNE_MAX_EPOCHS)
        
        class_loss, domain_loss, train_acc, domain_acc = train_one_epoch_finetune_dann(
            model=model, source_loader=source_train_loader,
            target_loader=target_train_loader, optimizer=optimizer,
            device=device, epoch=epoch, total_epochs=FINETUNE_MAX_EPOCHS,
            accumulation_steps=ACCUMULATION_STEPS, class_weights=class_weights,
            lambda_class=LAMBDA_CLASS, lambda_domain=current_lambda_domain
        )

        source_val_loss, source_val_acc, source_control_acc, source_dementia_acc, source_f1 = validate(
            model, source_val_loader, device, domain_name=SOURCE_DATASET
        )
        
        target_val_loss, target_val_acc, target_control_acc, target_dementia_acc, target_f1 = validate(
            model, target_val_loader, device, domain_name=TARGET_DATASET
        )
        
        avg_val_acc = source_weight * source_val_acc + target_weight * target_val_acc

        history['epochs'].append(epoch)
        history['train_class_losses'].append(class_loss)
        history['train_domain_losses'].append(domain_loss)
        history['train_accs'].append(train_acc)
        history['domain_accs'].append(domain_acc)
        history['source_val_accs'].append(source_val_acc)
        history['target_val_accs'].append(target_val_acc)
        history['avg_val_accs'].append(avg_val_acc)
        history['source_val_losses'].append(source_val_loss)
        history['target_val_losses'].append(target_val_loss)
        history['lambda_domains'].append(current_lambda_domain)

        print(f"Epoch {epoch:2d}/{FINETUNE_MAX_EPOCHS} | Train: {train_acc:.3f} | "
              f"Source: {source_val_acc:.3f} | Target: {target_val_acc:.3f} | "
              f"Avg: {avg_val_acc:.3f} | Domain: {domain_acc:.3f} | λ_d: {current_lambda_domain:.3f}")

        if (avg_val_acc > best_avg_acc and 
            source_val_acc >= min_source_acc and 
            target_val_acc >= min_target_acc):
            
            best_avg_acc = avg_val_acc
            best_epoch = epoch
            best_metrics = {
                'avg_val_acc': avg_val_acc, 'source_val_acc': source_val_acc,
                'target_val_acc': target_val_acc, 'source_val_loss': source_val_loss,
                'target_val_loss': target_val_loss, 'source_control_acc': source_control_acc,
                'source_dementia_acc': source_dementia_acc, 'target_control_acc': target_control_acc,
                'target_dementia_acc': target_dementia_acc, 'source_f1': source_f1,
                'target_f1': target_f1, 'epoch': epoch
            }
            patience_counter = 0
            
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch, 'best_avg_acc': best_avg_acc,
            }, seed_dir / 'best_model.pth')
            
            print(f"  *** Best model saved! Avg: {avg_val_acc*100:.2f}% "
                  f"(Source: {source_val_acc*100:.2f}%, Target: {target_val_acc*100:.2f}%) ***")
        else:
            patience_counter += 1

        if patience_counter >= FINETUNE_PATIENCE:
            print(f"\nEarly stopping triggered after {FINETUNE_PATIENCE} epochs without improvement")
            break

    print(f"\n{'='*70}")
    print(f"Training completed for seed {seed}")
    if best_metrics:
        print(f"Best Epoch: {best_metrics['epoch']}")
        print(f"Average Validation Accuracy: {best_metrics['avg_val_acc']*100:.2f}%")
        print(f"{SOURCE_DATASET}: Acc={best_metrics['source_val_acc']*100:.2f}%, F1={best_metrics['source_f1']:.4f}")
        print(f"{TARGET_DATASET}: Acc={best_metrics['target_val_acc']*100:.2f}%, F1={best_metrics['target_f1']:.4f}")
    print(f"{'='*70}\n")

    return seed, best_metrics, history


## Train with Multiple Seeds


In [12]:
all_results = {
    'seeds': [],
    'metrics': [],
    'histories': []
}


### Seed 21


In [ ]:
seed, metrics, history = train_finetune_dann(
    seed=21,
    source_train_loader=source_train_loader,
    target_train_loader=target_train_loader,
    source_val_loader=source_val_loader,
    target_val_loader=target_val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device,
    source_weight=SOURCE_WEIGHT,
    target_weight=TARGET_WEIGHT,
    min_source_acc=MIN_SOURCE_ACC,
    min_target_acc=MIN_TARGET_ACC
)

all_results['seeds'].append(seed)
all_results['metrics'].append(metrics)
all_results['histories'].append(history)

# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history['epochs'], history['train_class_losses'], label='Class Loss')
axes[0, 0].plot(history['epochs'], history['train_domain_losses'], label='Domain Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Losses')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history['epochs'], [acc*100 for acc in history['train_accs']], label='Train')
axes[0, 1].plot(history['epochs'], [acc*100 for acc in history['source_val_accs']], label='Source Val')
axes[0, 1].plot(history['epochs'], [acc*100 for acc in history['target_val_accs']], label='Target Val')
axes[0, 1].plot(history['epochs'], [acc*100 for acc in history['avg_val_accs']], label='Avg Val', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].set_title('Accuracy Curves')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history['epochs'], [acc*100 for acc in history['domain_accs']], color='purple')
axes[1, 0].axhline(y=50, color='r', linestyle='--', alpha=0.5, label='Random (50%)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Domain Accuracy (%)')
axes[1, 0].set_title('Domain Classifier Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history['epochs'], history['lambda_domains'], color='green')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Lambda Value')
axes[1, 1].set_title('Domain Loss Weight Schedule')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


XLSR:Using original XLSR model


In [ ]:
torch.cuda.empty_cache()


### Seed 42


In [ ]:
seed, metrics, history = train_finetune_dann(
    seed=42, source_train_loader=source_train_loader, target_train_loader=target_train_loader,
    source_val_loader=source_val_loader, target_val_loader=target_val_loader,
    output_dir=MODEL_OUTPUT_DIR, device=device,
    source_weight=SOURCE_WEIGHT, target_weight=TARGET_WEIGHT,
    min_source_acc=MIN_SOURCE_ACC, min_target_acc=MIN_TARGET_ACC
)
all_results['seeds'].append(seed)
all_results['metrics'].append(metrics)
all_results['histories'].append(history)
torch.cuda.empty_cache()


### Seed 84


In [ ]:
seed, metrics, history = train_finetune_dann(
    seed=84, source_train_loader=source_train_loader, target_train_loader=target_train_loader,
    source_val_loader=source_val_loader, target_val_loader=target_val_loader,
    output_dir=MODEL_OUTPUT_DIR, device=device,
    source_weight=SOURCE_WEIGHT, target_weight=TARGET_WEIGHT,
    min_source_acc=MIN_SOURCE_ACC, min_target_acc=MIN_TARGET_ACC
)
all_results['seeds'].append(seed)
all_results['metrics'].append(metrics)
all_results['histories'].append(history)
torch.cuda.empty_cache()


### Seed 168


In [ ]:
seed, metrics, history = train_finetune_dann(
    seed=168, source_train_loader=source_train_loader, target_train_loader=target_train_loader,
    source_val_loader=source_val_loader, target_val_loader=target_val_loader,
    output_dir=MODEL_OUTPUT_DIR, device=device,
    source_weight=SOURCE_WEIGHT, target_weight=TARGET_WEIGHT,
    min_source_acc=MIN_SOURCE_ACC, min_target_acc=MIN_TARGET_ACC
)
all_results['seeds'].append(seed)
all_results['metrics'].append(metrics)
all_results['histories'].append(history)
torch.cuda.empty_cache()


### Seed 336


In [ ]:
seed, metrics, history = train_finetune_dann(
    seed=336, source_train_loader=source_train_loader, target_train_loader=target_train_loader,
    source_val_loader=source_val_loader, target_val_loader=target_val_loader,
    output_dir=MODEL_OUTPUT_DIR, device=device,
    source_weight=SOURCE_WEIGHT, target_weight=TARGET_WEIGHT,
    min_source_acc=MIN_SOURCE_ACC, min_target_acc=MIN_TARGET_ACC
)
all_results['seeds'].append(seed)
all_results['metrics'].append(metrics)
all_results['histories'].append(history)
torch.cuda.empty_cache()


## Summary and Analysis


In [ ]:
# Extract results
seeds = all_results['seeds']
metrics_list = all_results['metrics']

# Filter out empty metrics
valid_metrics = [m for m in metrics_list if m]

if valid_metrics:
    source_accs = [m['source_val_acc'] * 100 for m in valid_metrics]
    source_f1s = [m['source_f1'] for m in valid_metrics]
    source_control_accs = [m['source_control_acc'] * 100 for m in valid_metrics]
    source_dementia_accs = [m['source_dementia_acc'] * 100 for m in valid_metrics]
    
    target_accs = [m['target_val_acc'] * 100 for m in valid_metrics]
    target_f1s = [m['target_f1'] for m in valid_metrics]
    target_control_accs = [m['target_control_acc'] * 100 for m in valid_metrics]
    target_dementia_accs = [m['target_dementia_acc'] * 100 for m in valid_metrics]
    
    avg_accs = [m['avg_val_acc'] * 100 for m in valid_metrics]
    
    print(f"{'='*80}")
    print(f"SUMMARY: Fine-tuned DANN - {SOURCE_DATASET} → {TARGET_DATASET}")
    print(f"{'='*80}")
    print(f"\nMean Results across {len(valid_metrics)} seeds:")
    print(f"{'-'*80}")
    print(f"{'Metric':<30} {'Source (Pitt)':<25} {'Target (Lu)':<25}")
    print(f"{'-'*80}")
    print(f"{'Overall Accuracy':<30} {np.mean(source_accs):>6.2f}% ± {np.std(source_accs):>4.2f}%     {np.mean(target_accs):>6.2f}% ± {np.std(target_accs):>4.2f}%")
    print(f"{'F1 Score':<30} {np.mean(source_f1s):>8.4f} ± {np.std(source_f1s):>6.4f}   {np.mean(target_f1s):>8.4f} ± {np.std(target_f1s):>6.4f}")
    print(f"{'Control Accuracy':<30} {np.mean(source_control_accs):>6.2f}% ± {np.std(source_control_accs):>4.2f}%     {np.mean(target_control_accs):>6.2f}% ± {np.std(target_control_accs):>4.2f}%")
    print(f"{'Dementia Accuracy':<30} {np.mean(source_dementia_accs):>6.2f}% ± {np.std(source_dementia_accs):>4.2f}%     {np.mean(target_dementia_accs):>6.2f}% ± {np.std(target_dementia_accs):>4.2f}%")
    print(f"{'-'*80}")
    print(f"{'Average Validation Accuracy:':<30} {np.mean(avg_accs):>6.2f}% ± {np.std(avg_accs):>4.2f}%")
    print(f"{'Performance Gap:':<30} {np.mean(source_accs) - np.mean(target_accs):>6.2f}%")
    print(f"{'='*80}")
    
    print(f"\nDetailed Results by Seed:")
    print(f"{'-'*80}")
    for i, (seed, m) in enumerate(zip(seeds, metrics_list)):
        if m:
            print(f"\nSeed {seed}:")
            print(f"  {SOURCE_DATASET:>8} - Acc: {m['source_val_acc']*100:>6.2f}%, F1: {m['source_f1']:>6.4f}")
            print(f"  {TARGET_DATASET:>8} - Acc: {m['target_val_acc']*100:>6.2f}%, F1: {m['target_f1']:>6.4f}")
            print(f"  Average  - {m['avg_val_acc']*100:>6.2f}%")
    print(f"{'-'*80}")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(valid_metrics))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, source_accs, width, label=f'{SOURCE_DATASET} (Source)', color='#3498db')
    bars2 = ax.bar(x + width/2, target_accs, width, label=f'{TARGET_DATASET} (Target)', color='#e74c3c')
    
    ax.set_xlabel('Seed', fontsize=12, fontweight='bold')
    ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Fine-tuned DANN: {SOURCE_DATASET} → {TARGET_DATASET}', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([seeds[i] for i in range(len(valid_metrics))])
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_axisbelow(True)
    
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("No seeds met the minimum accuracy thresholds.")
    print(f"Try lowering MIN_SOURCE_ACC ({MIN_SOURCE_ACC}) and MIN_TARGET_ACC ({MIN_TARGET_ACC})")


In [ ]:
print("\n" + "="*80)
print("METHOD COMPARISON")
print("="*80)
print("\nApproach Comparison:")
print(f"{'Method':<30} {'XLSR Status':<20} {'Domain Adaptation':<20}")
print("-"*80)
print(f"{'Finetuned-XLSR':<30} {'Partially trainable':<20} {'No':<20}")
print(f"{'Frozen-DANN':<30} {'Completely frozen':<20} {'Yes':<20}")
print(f"{'Finetuned-DANN (This)':<30} {'Partially trainable':<20} {'Yes':<20}")
print("="*80)

if valid_metrics:
    print(f"\nFinetuned-DANN Results:")
    print(f"  Source Domain ({SOURCE_DATASET}): {np.mean(source_accs):.2f}% ± {np.std(source_accs):.2f}%")
    print(f"  Target Domain ({TARGET_DATASET}): {np.mean(target_accs):.2f}% ± {np.std(target_accs):.2f}%")
    print(f"  Performance Gap: {np.mean(source_accs) - np.mean(target_accs):.2f}%")
    print(f"\nAdvantages:")
    print(f"  ✓ Fine-tuning enables better feature adaptation")
    print(f"  ✓ DANN reduces domain shift")
    print(f"  ✓ Should achieve better cross-domain generalization than either method alone")
    print(f"\nExpected Improvements:")
    print(f"  - Better source domain performance than Frozen-DANN (due to fine-tuning)")
    print(f"  - Better target domain performance than Finetuned-XLSR (due to domain adaptation)")
    print(f"  - Smaller performance gap between source and target domains")
